In [1]:
import polars as pl
import sys
import os
import datetime as dt
sys.path.append(os.path.abspath('..'))

In [2]:
from src_strategy.data_ingest.dataloader_1 import DataLoader
from src_strategy.configs.dataconfig import input_output_config_3, bp_config, data_config
from src_strategy.configs.outlierdetection.extremeoutliers import flowsheet_bounds_config, lab_bounds_config, bp_bounds_config

### Data loading

In [3]:
dl = DataLoader(
	input_output_config_3,
	data_config,
	bp_config,
	flowsheet_bounds_config,
	lab_bounds_config,
	bp_bounds_config
)


flowsheets = dl._load_flowsheets()
lab_res = dl._load_labs()
med_admin = dl._load_meds()
procedures = dl._load_procedures()
encounters = dl._load_encounters()
diagnosis = dl._load_diagnoses()

print(f'Flowsheets rows: {flowsheets.height}')
flowsheets = flowsheets.unique(subset=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value', 'Flag'])
print(f'Flowsheets rows after removing duplicates: {flowsheets.height}')

print(f'Labs rows: {lab_res.height}')
lab_res = lab_res.unique(subset=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value', 'Flag'])
print(f'Labs rows after removing duplicates: {lab_res.height}')


print(f'Meds rows: {med_admin.height}')
med_admin = med_admin.unique(subset=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value', 'Flag'])
print(f'Meds rows after removing duplicates: {med_admin.height}')

print(f'procedures rows: {procedures.height}')
procedures = procedures.unique(subset=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value', 'Flag'])
print(f'procedures rows after removing duplicates: {procedures.height}')

transform_dict = dl._define_transformation_pipelines()

logger =None
flowsheets_preprocessed = transform_dict['flowsheets'].run(flowsheets, logger)
labs_preprocessed = transform_dict['labs'].run(lab_res, logger)
med_admin = transform_dict['meds'].run(med_admin, logger)
procedures = transform_dict['procedures'].run(procedures, logger)
diagnosis = transform_dict['diagnoses'].run(diagnosis, logger)
encounters = transform_dict['encounters'].run(encounters, logger)

Flowsheets rows: 19007903
Flowsheets rows after removing duplicates: 19007903
Labs rows: 1988363
Labs rows after removing duplicates: 1978664
Meds rows: 985705
Meds rows after removing duplicates: 979429
procedures rows: 130186
procedures rows after removing duplicates: 87929


AttributeError: 'DataLoader' object has no attribute 'pf_config'

In [10]:
df_vent_on_off = flowsheets.filter(
    pl.col("Event_Grouper").is_in(["Vent on Documentation", "Vent off Documentation"])
).with_columns(
    pl.col("EncounterEpicCsn").cast(pl.Int64).alias("EncounterEpicCsn"),
	pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f").alias("Event_DateTime")
)
df_vent_on_off = df_vent_on_off.sort(by=['EncounterEpicCsn', 'Event_DateTime'])


/tmp/ipykernel_502923/3795625877.py:5: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  pl.col("Event_DateTime").str.strptime(dtype=pl.Datetime, format="%Y-%m-%d %H:%M:%S.%f").alias("Event_DateTime")


In [ ]:
df_vent_on = df_vent_on_off.filter(pl.col("Event_Grouper").is_in(["Vent on Documentation"]))
df_vent_off = df_vent_on_off.filter(pl.col("Event_Grouper").is_in(["Vent off Documentation"]))


In [15]:
df_vent_on

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,str,str,str
659308243,2022-06-14 21:51:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
659308243,2022-06-17 19:54:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
660956097,2022-06-28 05:10:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
661424247,2022-08-28 05:15:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
662358356,2022-06-13 05:32:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
…,…,…,…,…,…,…,…
734874042,2025-09-08 19:00:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
734886386,2025-09-05 13:39:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null
734912826,2025-09-06 01:30:00,"""Flowsheet""","""Vent on Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ Initial""",null


In [16]:
df_vent_off

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,str,str,str
661424247,2022-09-11 07:00:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
662358356,2022-06-13 13:08:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
662715699,2022-06-30 14:09:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
663020918,2022-06-08 17:48:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
663117991,2022-06-18 07:20:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
…,…,…,…,…,…,…,…
752744697,2026-05-13 22:05:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
752771158,2026-05-20 12:00:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null
753113388,2026-05-25 17:45:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null


In [18]:
df_vent_off_sub = df_vent_off.select("EncounterEpicCsn", pl.col("Event_DateTime").alias("off_time"), pl.col("Value"))
df_vent_on_sub = df_vent_on.select("EncounterEpicCsn", pl.col("Event_DateTime").alias("on_time"), pl.col("Value"))

In [58]:
df_vent_off_forward_on = df_vent_off_sub.join_asof(
    df_vent_on_sub,
    left_on='off_time',
	right_on='on_time',
	by='EncounterEpicCsn',
	strategy='backward'
)

/tmp/ipykernel_502923/4128773231.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_vent_off_forward_on = df_vent_off_sub.join_asof(


In [59]:
df_vent_off_forward_on['on_time'].null_count()/len(df_vent_off_forward_on)

0.2020746887966805

In [60]:
df_vent_off_forward_on.with_columns(
    (pl.col("off_time")-pl.col("on_time")).dt.total_hours(fractional=True).alias("vent_duration_hrs")
)['vent_duration_hrs']

vent_duration_hrs
f64
337.75
7.6
22.983333
138.633333
2.833333
…
null
null
null


In [67]:
flowsheets.filter(
    pl.col("Event_Grouper").str.to_lowercase().str.contains("vent")
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
str,str,str,str,str,str,str,str
"""684625198""","""2023-07-31 19:34:00.0000000""","""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null
"""677085552""","""2023-03-21 15:10:00.0000000""","""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null
"""737206914""","""2025-12-09 09:00:00.0000000""","""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null
"""704363461""","""2024-05-25 11:00:00.0000000""","""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null
"""706016401""","""2024-07-04 09:57:00.0000000""","""Flowsheet""","""O2 Delivery Mechanical Ventila…","""CPM S25 R INV DEVICE.INV O2 DE…",null,"""ventilator""",null
…,…,…,…,…,…,…,…
"""699476058""","""2025-03-14 14:46:00.0000000""","""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null
"""691998480""","""2023-11-04 02:55:00.0000000""","""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""$ On Going Hospital Vent""",null
"""740747803""","""2025-11-29 13:23:00.0000000""","""Flowsheet""","""Vent On/Off""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""On Going Hospital Vent""",null


In [66]:
df_vent_on_off.filter(
    (pl.col("EncounterEpicCsn") == 663459597)
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,str,str,str
663459597,2022-06-26 17:40:00,"""Flowsheet""","""Vent off Documentation""","""UTSW UH R RT VENT MANAGEMENT S…",null,"""Discontinued""",null


In [65]:
df_vent_off_sub.join_asof(
    df_vent_on_sub, 
	left_on='off_time',
	right_on='on_time',
	by='EncounterEpicCsn',
	strategy='nearest'
).filter(pl.col("on_time").is_null())

/tmp/ipykernel_502923/2589106938.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_vent_off_sub.join_asof(


EncounterEpicCsn,off_time,Value,on_time,Value_right
i64,datetime[μs],str,datetime[μs],str
663459597,2022-06-26 17:40:00,"""Discontinued""",null,null
663679803,2022-07-17 16:02:00,"""Discontinued""",null,null
674489746,2022-12-27 23:28:00,"""Discontinued""",null,null
674492790,2022-12-28 18:00:00,"""Discontinued""",null,null
674753933,2023-01-02 03:00:00,"""Discontinued""",null,null
…,…,…,…,…
752744697,2026-05-13 22:05:00,"""Discontinued""",null,null
752771158,2026-05-20 12:00:00,"""Discontinued""",null,null
753113388,2026-05-25 17:45:00,"""Discontinued""",null,null


Build a full-stack campaign concept studio for marketing teams using the current OpenAI Responses API.

The app should let a user enter a short campaign brief, target audience, product details, tone, and desired channels. It should generate:
- a concise campaign concept
- 3 headline/body copy variants
- a launch checklist
- image prompts and generated images for the campaign direction

Requirements:
- Use the current OpenAI API patterns for the Responses API, not legacy Completions or Chat Completions code.
- Use text generation and image generation in the flow.
- Create a clean, production-quality UI with loading, error, and empty states.
- Keep server-side OpenAI calls off the client and document the client/server boundary.
- Include OPENAI_API_KEY environment variable setup.
- Add a README with install, run, and deployment notes.
- Add a small validation plan and explain where to adjust the model, prompt, and image settings later.

Documentation:
- If the OpenAI Docs skill is available, use it to verify the latest OpenAI API guidance before implementing.
- If the OpenAI Docs skill is not available, install the OpenAI Docs skill and use it.
- If the OpenAI Docs skill cannot be installed or used, use web search to search the latest OpenAI developer documentation on developers.openai.com/api.
- Reference https://developers.openai.com/api/docs/models for guidance on the latest models to use.

Frontend:
- If the Frontend skill is installed, use it for frontend implementation and polish.
- If the Frontend skill is not installed, install the Frontend skill and use it.

In [53]:
df_vent_on_forward_off = df_vent_on_sub.join_asof(
    df_vent_off_sub,
	right_on='off_time',
	left_on='on_time',
	by='EncounterEpicCsn',
	strategy='forward'
)

/tmp/ipykernel_502923/2746336856.py:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_vent_on_forward_off = df_vent_on_sub.join_asof(


In [54]:
df_vent_on_forward_off['off_time'].null_count()/len(df_vent_on_forward_off)

0.21159190154823343

In [57]:
df_vent_on_forward_off.with_columns(
    (pl.col("off_time")-pl.col("on_time")).dt.total_hours(fractional=True).alias("vent_duration_hrs")
)

EncounterEpicCsn,on_time,Value,off_time,Value_right,vent_duration_hrs
i64,datetime[μs],str,datetime[μs],str,f64
659308243,2022-06-14 21:51:00,"""$ Initial""",null,null,null
659308243,2022-06-17 19:54:00,"""$ Initial""",null,null,null
660956097,2022-06-28 05:10:00,"""$ Initial""",null,null,null
661424247,2022-08-28 05:15:00,"""$ Initial""",2022-09-11 07:00:00,"""Discontinued""",337.75
662358356,2022-06-13 05:32:00,"""$ Initial""",2022-06-13 13:08:00,"""Discontinued""",7.6
…,…,…,…,…,…
734874042,2025-09-08 19:00:00,"""$ Initial""",2025-09-09 05:40:00,"""Discontinued""",10.666667
734886386,2025-09-05 13:39:00,"""$ Initial""",2025-09-08 11:30:00,"""Discontinued""",69.85
734912826,2025-09-06 01:30:00,"""$ Initial""",null,null,null


In [6]:
encounters['Baseline_eGFR'].null_count()/len(encounters)

0.20893120557865094

In [4]:
flowsheets_preprocessed.columns

['EncounterEpicCsn',
 'Event_DateTime',
 'Type',
 'Event_Grouper',
 'Event_Name',
 'NumericValue',
 'Value',
 'Flag',
 'sys',
 'dia',
 'map',
 'flowsheet_outlier',
 'bp_outlier_sys',
 'bp_outlier_dia',
 'bp_outlier_map']

### Finding groupers with numerical values that occur at the same instant

In [ ]:
groupers_with_numerical_values = flowsheets_preprocessed.filter(
    pl.col("NumericValue").is_not_null()
)['Event_Grouper'].value_counts(sort=True)['Event_Grouper'].to_list()

groupers_with_multiple_values_at_same_instant = flowsheets_preprocessed.group_by(
	["EncounterEpicCsn", "Event_DateTime", "Event_Grouper"]
).agg(
    pl.len().alias('n_rows')
).filter(pl.col("n_rows")>1)['Event_Grouper'].unique().to_list()


(['Pulse',
  'Respirations',
  'Arterial Blood Pressure Mean',
  'Temperature',
  'Glasgow Coma Score',
  'O2 Flow Rate',
  'Braden Scale',
  'Weight',
  'BMI'],
 ['Oxygen Delivery',
  'Glasgow Coma Score',
  'O2 Delivery Non-Rebreather Mask',
  'O2 Delivery Mechanical Ventilation',
  'O2 Delivery Nasal Cannula'])

In [17]:
set(groupers_with_numerical_values).intersection(groupers_with_multiple_values_at_same_instant)

{'Glasgow Coma Score'}

In [18]:
groupers_with_numerical_values = labs_preprocessed.filter(
    pl.col("NumericValue").is_not_null()
)['Event_Grouper'].value_counts(sort=True)['Event_Grouper'].to_list()

groupers_with_multiple_values_at_same_instant = labs_preprocessed.group_by(
	
	["EncounterEpicCsn", "Event_DateTime", "Event_Grouper"]
).agg(
    pl.len().alias('n_rows')
).filter(pl.col("n_rows")>1)['Event_Grouper'].unique().to_list()

In [19]:
set(groupers_with_numerical_values).intersection(groupers_with_multiple_values_at_same_instant)

{'Bilirubin',
 'Blood Urea Nitrogen',
 'Creatinine',
 'INR',
 'Lactate',
 'PAO2',
 'Platelets',
 'Sepsis Body Fluid Culture Orders',
 'WBC',
 'WBC in Urine',
 'WBC in Urine - Confirmed Infection',
 'eGFR'}

In [26]:
labs_preprocessed.filter(
    pl.col("Event_Grouper").is_in(
		set(groupers_with_numerical_values).intersection(groupers_with_multiple_values_at_same_instant)
	)
).group_by(
	["EncounterEpicCsn", "Event_DateTime", "Event_Grouper"]
).agg(
    pl.col("NumericValue").min().alias('min_val'),
    pl.col("NumericValue").max().alias('max_val')
).filter(
    pl.col("min_val")!=pl.col("max_val")
)

EncounterEpicCsn,Event_DateTime,Event_Grouper,min_val,max_val
i64,datetime[μs],str,f64,f64
695250741,2024-01-19 15:01:00,"""Blood Urea Nitrogen""",13.0,14.0
699449380,2024-03-04 14:47:00,"""Creatinine""",0.88,0.92
663380959,2022-07-10 17:20:00,"""Platelets""",49.0,50.0
666640816,2022-08-11 23:22:00,"""Bilirubin""",6.3,6.4
712742998,2024-09-30 04:48:00,"""eGFR""",142.0,145.0
…,…,…,…,…
692947899,2023-12-05 18:47:00,"""WBC""",5.69,5.82
691961409,2023-11-21 17:23:00,"""eGFR""",95.0,97.0
668008206,2022-09-10 09:50:00,"""Creatinine""",1.3,1.31


In [14]:
procedures.filter(
    pl.col("NumericValue").is_not_null()
)['Event_Grouper'].value_counts(sort=True)

Event_Grouper,count
str,u64


In [5]:
selected_cols = ['EncounterEpicCsn', 'Event_DateTime', "Type", "Event_Grouper", 'Event_Name', 'Value', 'Flag']
df_all = pl.concat(
    [
        flowsheets_preprocessed.select(selected_cols),
        labs_preprocessed.select(selected_cols),
        procedures.select(selected_cols),
        med_admin.select(selected_cols),
        diagnosis.select(selected_cols)
	],how='vertical'
)

In [6]:
df_all = df_all.sort(by=['EncounterEpicCsn', 'Event_DateTime'], descending=False)

In [7]:
df_all.shape

(23552400, 7)

In [49]:
groups = df_all.group_by(
    ["EncounterEpicCsn", "Event_DateTime", "Event_Grouper"]
).agg(
	pl.len().alias('n_rows')
).filter(pl.col("n_rows")>1).sort(by='n_rows', descending=True)

In [52]:
groups

EncounterEpicCsn,Event_DateTime,Event_Grouper,n_rows
i64,datetime[μs],str,u64
699476058,2025-11-03 01:39:25.337,"""Billing Diagnosis""",181
680127307,2023-09-01 10:45:28.053,"""Billing Diagnosis""",176
717006646,2025-06-10 02:24:07.240,"""Billing Diagnosis""",175
697718727,2024-12-12 03:45:04.590,"""Billing Diagnosis""",163
705630549,2024-11-23 04:11:41.037,"""Billing Diagnosis""",154
…,…,…,…
734939447,2025-11-21 16:06:00,"""Antibiotics""",2
703702500,2024-05-12 02:00:57.937,"""Problem List""",2
686564255,2023-08-08 09:48:00,"""Blood Culture - Confirmed Infe…",2


In [58]:
df_all.filter(
	(pl.col('EncounterEpicCsn') == 699476058)&
	(pl.col('Event_DateTime') >= dt.datetime(2025, 11, 3, 1, 39, 25))&
	(pl.col('Event_DateTime') <= dt.datetime(2025, 11, 3, 1, 39, 26))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,Value,Flag
i64,datetime[μs],str,str,str,str,str
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Other chronic pain""","""G89.29""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Candidal sepsis (*)""","""B37.7""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Other ventricular tachycardia""","""I47.29""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Procedure and treatment not ca…","""Z53.20""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Critical illness myopathy""","""G72.81""",null
…,…,…,…,…,…,…
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Other general symptoms and sig…","""R68.89""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Pressure ulcer of sacral regio…","""L89.154""",null
699476058,2025-11-03 01:39:25.337,"""Diagnosis Event""","""Billing Diagnosis""","""Other sites of candidiasis""","""B37.89""",null


### Monitor same instant collision

In [10]:
labs_groups = labs_preprocessed.group_by(
    "EncounterEpicCsn", "Event_DateTime", "Event_Grouper"
).agg(
    pl.len().alias('n_rows')
)

In [15]:
labs_groups.filter(pl.col('n_rows')>1).sort(by='n_rows', descending=True)

EncounterEpicCsn,Event_DateTime,Event_Grouper,n_rows
i64,datetime[μs],str,u64
682432036,2023-05-21 03:25:00,"""Blood Culture - Confirmed Infe…",13
718905739,2025-01-06 15:52:00,"""Blood Culture - Confirmed Infe…",9
674214921,2022-12-20 06:09:00,"""Blood Culture - Confirmed Infe…",7
683544133,2023-10-06 05:38:00,"""Sepsis Body Fluid Culture Orde…",6
707657122,2024-08-26 11:51:00,"""Sepsis Body Fluid Culture Orde…",6
…,…,…,…
668763648,2022-09-17 13:11:00,"""Blood Culture""",2
672725664,2022-11-22 09:31:00,"""eGFR""",2
693789029,2024-01-18 12:24:00,"""Blood Culture""",2


In [ ]:
labs_preprocessed.filter(
    (pl.col('EncounterEpicCsn') == 682432036)&
    (pl.col('Event_DateTime') == dt.datetime(2023, 5, 21, 3, 25) )
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,lab_outlier
i64,datetime[μs],str,str,str,f64,str,str,f64


In [19]:
labs_preprocessed.filter(
    (pl.col('EncounterEpicCsn') == 683544133)&
    (pl.col('Event_DateTime') == dt.datetime(2023, 10, 6, 5, 38))
).sort(by='Event_DateTime')

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag,lab_outlier
i64,datetime[μs],str,str,str,f64,str,str,f64
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""CULTURE""",null,""">=10,000 Colonies/cc Capnocyto…","""1""",null
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GRAM ST""",null,""">25 PER LOW POWER FIELD WBCs o…","""1""",null
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GRAM ST""",null,"""<10 PER LOW POWER FIELD Squamo…","""1""",null
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GRAM ST""",null,""">25 PER LOW POWER FIELD WBCs o…","""1""",null
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""GRAM ST""",null,"""<10 PER LOW POWER FIELD Squamo…","""1""",null
683544133,2023-10-06 05:38:00,"""Lab Results""","""Sepsis Body Fluid Culture Orde…","""CULTURE""",null,""">=10,000 Colonies/cc Capnocyto…","""1""",null


In [20]:
labs_preprocessed.shape

(1988363, 9)

In [28]:
labs_preprocessed.unique(
    subset=['EncounterEpicCsn', 'Event_DateTime', 'Event_Name', 'Value']
).shape

(1978659, 9)

In [34]:
procedures_groups = procedures.group_by(
    "EncounterEpicCsn", "Event_DateTime", "Event_Grouper"
).agg(
    pl.len().alias('n_rows')
)

In [36]:
procedures_groups.filter(pl.col("n_rows")>1).sort(by='n_rows', descending=True)

EncounterEpicCsn,Event_DateTime,Event_Grouper,n_rows
i64,datetime[μs],str,u64
700432870,2024-03-20 02:46:00,"""Lactate""",8
734030770,2026-03-10 09:42:00,"""Blood Culture Order""",8
699476058,2024-04-30 21:55:00,"""Lactate""",7
672929142,2022-12-20 17:38:00,"""Blood Culture Order""",6
685993148,2023-07-26 14:37:00,"""Lactate""",6
…,…,…,…
710269921,2024-08-28 13:04:00,"""Blood Culture Order""",2
742482436,2025-12-18 11:38:00,"""Blood Culture Order""",2
688651091,2024-01-23 11:39:00,"""Blood Culture Order""",2


In [40]:
procedures.filter(
    (pl.col("EncounterEpicCsn")==700432870)&
    (pl.col("Event_DateTime")==dt.datetime(2024, 3, 20, 2, 46))
)

EncounterEpicCsn,Event_DateTime,Type,Event_Grouper,Event_Name,NumericValue,Value,Flag
i64,datetime[μs],str,str,str,f64,str,str
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Lactate""","""LACTATE (LACTIC ACID) SERUM""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Lactate""","""LACTATE (LACTIC ACID) SERUM""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Lactate""","""LACTATE (LACTIC ACID) SERUM""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Lactate""","""LACTATE (LACTIC ACID) SERUM""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Lactate""","""LACTATE (LACTIC ACID) SERUM""",null,null,null
…,…,…,…,…,…,…,…
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Code Sepsis Page""","""UTSW CODE SEPSIS PAGE""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Code Sepsis Page""","""UTSW CODE SEPSIS PAGE""",null,null,null
700432870,2024-03-20 02:46:00,"""Procedure Order""","""Code Sepsis Page""","""UTSW CODE SEPSIS PAGE""",null,null,null
